# Scenario: The Danger of Default "No Allergies"

In [14]:
import pandas as pd
import sqlite3
# creating dataset representing allergy logs over two weeks
allergy_data = {
    "entry_id": [101, 102, 103, 104, 105, 106, 107, 108],
    "input_date": ["2026-05-10", "2026-05-10", "2026-05-11", "2026-05-12", 
                  "2026-05-15", "2026-05-15", "2026-05-16", "2026-05-16"],
    "staff_id": ["Nurse_A", "Nurse_B", "Nurse_A", "Nurse_C", "System_Default", "System_Default", "System_Default", "Nurse_B"],
    "allergy_status": ["Penicillin", "Peanuts", "None", "Sulfa", "None", "None", "None", "Latex"]
}
# adding dataset into the DataFrame
df_allergy_data = pd.DataFrame(allergy_data)
# creating sql to save the dataset in the temp memory
connt = sqlite3.connect(":memory:")
df_allergy_data. to_sql("allergy_records", connt, index = False, if_exists = "replace")
# function to run the query
def run_query(query):
    return pd.read_sql_query(query, connt)
print("*********************************** Default Value Proliferation Database is ready! **************")

*********************************** Default Value Proliferation Database is ready! **************


# Isolating the System-Generated Skew

In [15]:
# query for all data to review
all_data = "SELECT * FROM allergy_records"
print("********************************** all data for review **********************")
print()
display(run_query(all_data))
# query to count how many times "None" appears as the allergy_status, grouped by the staff_id that entered it.
total_none_value = """
SELECT staff_id, COUNT(*) AS none_count  
FROM allergy_records
WHERE allergy_status = 'None'
GROUP BY staff_id
"""
print("************************************ the total none values ********************")
display(run_query(total_none_value))

********************************** all data for review **********************



,entry_id,input_date,staff_id,allergy_status
0,101,2026-05-10,Nurse_A,Penicillin
1,102,2026-05-10,Nurse_B,Peanuts
2,103,2026-05-11,Nurse_A,None
3,104,2026-05-12,Nurse_C,Sulfa
4,105,2026-05-15,System_Default,None
5,106,2026-05-15,System_Default,None
6,107,2026-05-16,System_Default,None
7,108,2026-05-16,Nurse_B,Latex


************************************ the total none values ********************


,staff_id,none_count
0,Nurse_A,1
1,System_Default,3


# Calculating the Proliferation Rate

In [16]:
# query that calculates the total number of entries in the database and what percentage of those total entries are attributed to a System_Default entering "None"
system_entry_rate_as_none = """
SELECT
    COUNT(*) AS total_records,
    SUM(CASE WHEN staff_id = 'System_Default' AND allergy_status = 'None' THEN 1 ELSE 0 END) AS default_none_count,
    (CAST(SUM(CASE WHEN staff_id = 'System_Default' AND allergy_status = 'None' THEN 1 ELSE 0 END) AS REAL) / COUNT(*)) * 100 AS default_percentage
FROM allergy_records
"""
print("********************************* Percentage of the total entries by the system *******************")
display(run_query(system_entry_rate_as_none))

********************************* Percentage of the total entries by the system *******************


,total_records,default_none_count,default_percentage
0,8,3,37.5
